## Prepare Workspace

### Import Packages

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

### Set File Paths

In [ ]:
# Define user
user = os.getlogin()

# Working directories
path_sp  = os.path.join('C:\\Users', 'jfontes', 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', 'jfontes', 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# Set file paths
path_config = os.path.join(path_git, 'Pipeline', 'Python Code', 'Census', 'aa_config')
path_out    = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')


### Source User Defined Functions/Objects

In [ ]:
## User defined functions
exec(open(os.path.join(path_config, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

## Prepare Inputs

### Set Estimate Parameters

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'Parameters')
df_params = df_params[df_params['Run'] == 1]

# Set parameters for querying ACS data
estimate    = df_params['estimate' ].values[0]
sample_type = df_params['sample'   ].values[0]
geography   = df_params['geography'].values[0]

inputs = sample_type + '_' + geography
proportions = df_params['proportions'].values[0]

### Define Import Inputs

In [ ]:
## Import Variable Mapping
df_vars   = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = sample_type)
df_inputs = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = inputs)

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
year_start = int(df_inputs['year_start'].values[0])
year_end   = int(df_inputs['year_end'  ].values[0])
sp_folder_out = df_inputs['sp_folder'].values[0]

# Subset variables
df_vars = df_vars[df_vars['Indicator Name'] == indicator_name]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set years
years_to_import = list(range(year_start, year_end+1))

if estimate == 'ACS1':
    try:
        years_to_import.remove(2020)
    except:
        pass


if sample_type == 'ACS':
    # Set tables and variables to import
    list_vars = ['NAME'] + df_vars['ID'].to_list()
    tables = df_vars['Table'].unique()

    if (inputs == 'ACS_Tract') | (inputs == 'ACS_County'):
        
        # Import County FIPS mapping
        df_fips = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx')
                                , sheet_name = 'FIPSmap'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        
        df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
        
        # Convert to dictionary
        dict_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values)) 
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(tables)
    
    
    if inputs == 'ACS_MSA':
    
        # Set MSA
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)


if inputs == 'PUMS_PUMA':

    # Set PUMA variables
    record_type = df_inputs['record_type'].values[0]
    list_vars = ['PUMA'] + df_vars['ID'].to_list()
    variables = ','.join(list_vars)
    if record_type == 'P': 
        variables = ','.join([variables, 'PWGTP'])
    if record_type == 'H': 
        variables = ','.join([variables,  'WGTP'])

    # Import County FIPS mapping
    df_fips = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx')
                            , sheet_name = 'FIPSmap'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)] 
    states = df_fips['State FIPS'].unique()
    print(record_type)

# view
print(indicator_name)
print(year_start)
print(year_end)
df_vars.head()

In [ ]:
if sample_type == 'ACS':

    ## Build initial ID fields ##
    print("Building initial ID fields")
    print("")
          
    # initialize empty list to store data frames
    list_df_acs = []
    
    # only want one table
    df_table = df_vars[df_vars['Table'] == tables[0]]
    list_table_vars = ['NAME'] + df_table['ID'].to_list()[0:2]
    variables = ",".join(list_table_vars)
    
    
    if inputs == 'ACS_Tract':
        # pull data, subset to just ID fields
        for state in list(dict_fips.keys()):
            print('State: ' + state)
            for year in tqdm(years_to_import):
                try:
                    temp = acs5_state_county_tract(api_Key     = api_key
                                                   , variables = variables
                                                   , year      = year
                                                   , state     = state
                                                   , county    = dict_fips[state])
                    
                    temp = temp[['NAME', 'state', 'county', 'tract', 'Year']]
                    list_df_acs.append(temp)
                except:
                    pass
                    
    if inputs == 'ACS_County':
        # pull data, subset to just ID fields
        for state in list(dict_fips.keys()):
            print('State: ' + state)
            for year in tqdm(years_to_import):
                try:
                    if estimate == 'ACS1':
                        temp = acs1_state_county(api_Key     = api_key
                                                       , variables = variables
                                                       , year      = year
                                                       , state     = state
                                                       , county    = dict_fips[state])
                        
                        temp = temp[['NAME', 'state', 'county', 'Year']]
                        list_df_acs.append(temp)
                    
                    if estimate == 'ACS5':
                        temp = acs5_state_county(api_Key     = api_key
                                                       , variables = variables
                                                       , year      = year
                                                       , state     = state
                                                       , county    = dict_fips[state])
                        
                        temp = temp[['NAME', 'state', 'county', 'Year']]
                        list_df_acs.append(temp)
                except:
                    pass
                    
    if inputs == 'ACS_MSA':
        # pull all years and counties for each table
        for year in tqdm(years_to_import):
            try:
                temp =  acs5_msa(api_Key     = api_key
                                 , variables = variables
                                 , year      = year
                                 , msa       = msa_to_import)
                
                temp = temp[['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']]
                list_df_acs.append(temp)
            except:
                pass
                
    # combine all years and counties
    df_acs_raw = pd.concat(list_df_acs)
    
    if (inputs == 'ACS_Tract') | (inputs == 'ACS_County'):
        # merge county name onto table
        df_acs_raw = df_acs_raw.merge(df_fips[['County FIPS', 'County Name']], left_on = 'county', right_on = 'County FIPS')
        df_acs_raw.drop(['County FIPS'], axis = 1, inplace = True)
    
        
    print("Finished!")
    print("")
    
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling data from source")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    list_df_acs = []
    
    # iterate through each table (pulling all tables at once fails because the URL is too long - i think)
    for table in tables:
    
        # keep track of tables being imported
        print("")
        print("Table ID: " + table)
        list_df_tables = []
    
        # only want one table
        df_table = df_vars[df_vars['Table'] == table]
        list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+20] for x in range(0, len(df_table['ID'].to_list()), 20)]
        
        list_variables = []
        for x in list_table_vars:
            list_variables.append(",".join(x))
        
        # pull all years and counties for each table
        for variables in list_variables:
            print("Variables: " + variables)
            list_df_tables = []
                
            for year in tqdm(years_to_import):
                try:
                    if inputs == 'ACS_Tract':
                        list_df_tables.append(
                            acs5_state_county_tract(api_Key     = api_key
                                                    , variables = variables
                                                    , year      = year
                                                    , state     = state
                                                    , county    = dict_fips[state])
                        )

                    if inputs == 'ACS_County':
                        if estimate == 'ACS1':
                            list_df_tables.append(
                                acs1_state_county(api_Key     = api_key
                                                  , variables = variables
                                                  , year      = year
                                                  , state     = state
                                                  , county    = dict_fips[state])
                            )
                        
                        if estimate == 'ACS5':
                            list_df_tables.append(
                                acs5_state_county(api_Key     = api_key
                                                  , variables = variables
                                                  , year      = year
                                                  , state     = state
                                                  , county    = dict_fips[state])
                            )
                            
                            
                    if inputs == 'ACS_MSA':
                        if estimate == 'ACS1':
                            list_df_tables.append(
                                acs1_msa(api_Key     = api_key
                                         , variables = variables
                                         , year      = year
                                         , msa       = msa_to_import)
                            ) 
                            
                        if estimate == 'ACS5':
                            list_df_tables.append(
                                acs5_msa(api_Key     = api_key
                                         , variables = variables
                                         , year      = year
                                         , msa       = msa_to_import)
                            )
                except:
                    pass
    
            # combine all years and counties
            df_temp = pd.concat(list_df_tables)
    
            # left join data onto key
            if inputs == 'ACS_Tract':
                df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'left')
                
            if inputs == 'ACS_County':
                df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'Year'], how = 'left')
    
            if inputs == 'ACS_MSA':
                df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'left')
     
    print("Finished!")


if sample_type == 'PUMS':
    
    print("Importing and compiling data from source")
    print("")
    
    # initialize empty list to store data frames
    list_df_acs = []
    
    # pull all years into one table (takes 2-3 minutes per year)
    for state in states:
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                if estimate == 'PUMS1':
                    list_df_acs.append(
                        acs1_pums(api_Key       = api_key
                                  , variables   = variables
                                  , year        = year
                                  , state       = state
                                  , record_type = record_type)
                                )

                if estimate == 'PUMS5':
                    list_df_acs.append(
                        acs5_pums(api_Key       = api_key
                                  , variables   = variables
                                  , year        = year
                                  , state       = state
                                  , record_type = record_type)
                                )
            except:
                pass
    
    # combine all years
    df_acs_raw = pd.concat(list_df_acs)

    print("Finished!")

In [ ]:
# view raw data
pd.set_option('display.max_columns', None)
print(df_acs_raw.shape)
print(df_acs_raw.Year.unique())
df_acs_raw.head(3)

In [ ]:
df_acs = df_acs_raw.copy()
df_acs[list(df_inputs['groups'].values)] = df_acs[list(df_inputs['groups'].values)].apply(pd.to_numeric)
df_acs.describe()


In [ ]:
# make copy
df_acs = df_acs_raw.copy()

if inputs == 'ACS_Tract':
    # Melt data from wide to long
    df_acs = pd.melt(df_acs
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'tract', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    
    # Convert imported values to numeric
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    
    
    # Merge label 2
    df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    
    
    # Subset
    df_acs = df_acs[['ID', 'Table Name', 'Label', 'County Name', 'NAME', 'state', 
                     'county', 'tract', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    
    
    # Manually check column names and clean as needed
    df_acs = df_acs.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
        , 'tract':'Tract ID'
    })


if inputs == 'ACS_County':
    
    df_acs = df_acs.replace('null', np.nan)
    df_acs = df_acs.dropna(axis = 0, how = "any")
    
    # Melt data from wide to long
    df_acs = pd.melt(df_acs
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    
    # Convert imported values to numeric
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    
    
    # Merge label 2
    df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    
    
    # Subset
    df_acs = df_acs[['ID', 'Table Name', 'Label', 'County Name', 'NAME', 'state', 
                       'county', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    
    
    # Manually check column names and clean as needed
    df_acs = df_acs.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
    })



if inputs == 'ACS_MSA':
    
    df_acs = df_acs.rename(columns = {'metropolitan statistical area/micropolitan statistical area':'MSA_ID'})
    
    # mapping
    df_msa_map = df_acs[df_acs['Year'] == 2022][['NAME', 'MSA_ID']].drop_duplicates().rename(columns = {'NAME':'MSA'})
    df_acs = df_acs.merge(df_msa_map, on = 'MSA_ID')
    
    # Melt data from wide to long
    df_acs = pd.melt(df_acs
                      , id_vars = ['NAME', 'MSA', 'MSA_ID', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    
    # Convert imported values to numeric
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    
    
    # Merge label 2
    df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    
    
    # Subset
    df_acs = df_acs[['ID', 'Table Name', 'Label', 'MSA_ID', 'MSA', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]


if inputs == 'PUMS_PUMA':

    # Clean missing values
    

    # Convert weighted column to integer
    df_acs['PWGTP'] = df_acs['PWGTP'].astype(int)

    # Drop Record Type column
    df_acs = df_acs.drop(['RT'], axis = 1)


    # Import and merge Variable ID description
    df_acs[list(df_inputs['groups'].values)] = df_acs[list(df_inputs['groups'].values)].astype("string")

    df_puma_vars = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'PUMSvars')
    df_puma_vars['Value1'] = df_puma_vars['Value1'].astype("string")

    df_puma_vars = df_puma_vars.pivot_table(index = 'Value1'
                                           , columns = 'ID'
                                           , values = 'Description'
                                           , aggfunc = lambda x: x).reset_index()

    cols = ['Value1'] + list(df_inputs['groups'].values)
    df_puma_vars = df_puma_vars[cols]
    df_puma_vars = df_puma_vars.dropna()
    df_puma_vars = df_puma_vars.add_suffix('_desc').rename(columns = {'Value1_desc':'Value1'})

    for col in cols[1:]:
        df_acs = df_acs.merge(df_puma_vars[['Value1', col+'_desc']], left_on = col, right_on = 'Value1', how = 'left')
        df_acs = df_acs.drop(['Value1'], axis = 1)


df_acs.head()

In [ ]:
if sample_type == 'ACS':
    
    # Sort by census tract then by year then by race/ethnicity
    df_acs['Race_Ethnicity_sort'] = pd.Categorical(df_acs['Race_Ethnicity'], ['All'
                                                                     , 'AMERICAN INDIAN AND ALASKA NATIVE ALONE'
                                                                     , 'ASIAN ALONE'
                                                                     , 'BLACK OR FRICAN AMERICAN ALONE'
                                                                     , 'HISPANIC OR LATINO'
                                                                     , 'NATIVE HAWAIIAN AND OTHER PACIFIC ISLANDER ALONE'
                                                                     , 'WHITE ALONE'
                                                                     , 'WHITE ALONE, NOT HISPANIC OR LATINO'
                                                                     , 'SOME OTHER RACE ALONE'
                                                                     , 'TWO OR MORE RACES'])
    
    
    
    # sort and then remove categorical field
    
    if inputs == 'ACS_Tract':
        df_acs = df_acs.sort_values(by = ['NAME', 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = True)
    if inputs == 'ACS_MSA':
        df_acs = df_acs.sort_values(by = ['MSA', 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = True)
    
    df_acs = df_acs.drop(['Race_Ethnicity_sort', 'Sort'], axis = 1)
    df_acs.head()



if inputs == 'PUMS_PUMA':
    
    # df_acs = df_acs.merge(df_puma_names, on = ['PUMA'], how = 'left')

    # cols = ['state', 'PUMA', 'Year'] + list(df_acs.columns[1:-2])
    # # cols = ['state', 'PUMA', 'PUMA Name', 'Year'] + list(df_acs.columns[1:-3])
    # df_acs = df_acs[cols]
    
    # df_acs = df_acs.groupby(list(df_acs.columns[:-1]), as_index = False, sort = False)['PWGTP'].sum()
    
    # df_acs = df_acs.sort_values(list(df_acs.columns[:-1]))

In [ ]:
if inputs == 'ACS_Tract':
    # Create MPO and MSA groupings
    df_mpo = df_fips[['County Name', 'MPO']]    
    
    # Merge groupings
    df_acs = df_acs.merge(df_mpo, on = ['County Name'], how = 'left')    
    
    # reorder columns
    cols = ['Table ID', 'Table Name', 'Label', 'State FIPS', 'MPO', 'County Name', 
            'County FIPS', 'Tract ID', 'NAME', 'Year', 'Variable', 'Race_Ethnicity', 'Total']
    df_acs = df_acs[cols]
    
    
    # missing values represent a population of 0
    df_acs['Total'] = df_acs['Total'].fillna(0)

    
    ## Groupings roll up
    df_acs1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 'Tract ID', 'NAME', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()    
    
    # Counties
    df_counties1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 
                                   'Variable', 'Year', 'Race_Ethnicity'
                                  ], as_index = False, sort = False)['Total'].sum()
      
    
    # MPO
    df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()
    

    ## Dcasts
    # Tracts
    df_acs2 = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID'
                                           , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()

    
    # Counties
    df_counties2 = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS',
                                            'County Name', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()

    
   
    # MPO
    df_mpo2 = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()

    
    
    # missing values represent a population of 0
    df_acs2 = df_acs2.fillna(0)
    df_acs2 = df_acs2.fillna(0)
    df_mpo2 = df_mpo2.fillna(0)


    
    ## Check if we want to calculate proportions
    if proportions == 'Yes':
        
        # missing values represent a population of 0
        df_acs1     ['Proportion'] = df_acs1     ['Total'] / df_acs1[df_acs1          ['Variable'] != 'Total'].groupby(['NAME'      , 'Year'               ])['Total'].transform('sum')
        df_counties1['Proportion'] = df_counties1['Total'] / df_counties1[df_counties1['Variable'] != 'Total'].groupby(['State FIPS', 'County FIPS', 'Year'])['Total'].transform('sum')
        df_mpo1     ['Proportion'] = df_mpo1     ['Total'] / df_mpo1[df_mpo1          ['Variable'] != 'Total'].groupby(['MPO'       , 'Year'               ])['Total'].transform('sum')
        df_acs1     ['Proportion'] = df_acs1     ['Proportion'].fillna(1)
        df_counties1['Proportion'] = df_counties1['Proportion'].fillna(1)
        df_mpo1     ['Proportion'] = df_mpo1     ['Proportion'].fillna(1)

        # dcasts
        df_acs2_prop = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID', 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Proportion').reset_index()
        df_counties2_prop = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS', 'County Name', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Proportion').reset_index()
        df_mpo2_prop = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Proportion').reset_index()
        
        # missing values represent a population of 0
        df_acs2_prop = df_acs2_prop.fillna(0)
        df_acs2_prop = df_acs2_prop.fillna(0)
        df_mpo2_prop = df_mpo2_prop.fillna(0)


if inputs == 'ACS_County':
    # Create MPO and MSA groupings
    df_mpo = df_fips[['County Name', 'MPO']]    
    
    # Merge groupings
    df_acs = df_acs.merge(df_mpo, on = ['County Name'], how = 'left')    
    
    # reorder columns
    cols = ['Table ID', 'Table Name', 'Label', 'State FIPS', 'MPO', 'County Name', 
            'County FIPS', 'NAME', 'Year', 'Variable', 'Race_Ethnicity', 'Total']
    df_acs = df_acs[cols]
    
    
    # missing values represent a population of 0
    df_acs['Total'] = df_acs['Total'].fillna(0)

    
    ## Groupings roll up
    df_acs1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 'NAME', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()    
      
    
    # MPO
    df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()
    

    ## Dcasts
    # Tracts
    df_acs2 = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS'
                                           , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    
   
    # MPO
    df_mpo2 = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()

    
    
    # missing values represent a population of 0
    df_acs2 = df_acs2.fillna(0)
    df_mpo2 = df_mpo2.fillna(0)


    
    ## Check if we want to calculate proportions
    if proportions == 'Yes':
        
        # missing values represent a population of 0
        df_acs1['Proportion'] = df_acs1['Total'] / df_acs1[df_acs1['Variable'] != 'Total'].groupby(['NAME', 'Year'])['Total'].transform('sum')
        df_mpo1['Proportion'] = df_mpo1['Total'] / df_mpo1[df_mpo1['Variable'] != 'Total'].groupby(['MPO' , 'Year'])['Total'].transform('sum')
        df_acs1['Proportion'] = df_acs1['Proportion'].fillna(1)
        df_mpo1['Proportion'] = df_mpo1['Proportion'].fillna(1)

        # dcasts
        df_acs2_prop = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS',  'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Proportion').reset_index()
        df_mpo2_prop = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Proportion').reset_index()
        
        # missing values represent a population of 0
        df_acs2_prop = df_acs2_prop.fillna(0)
        df_mpo2_prop = df_mpo2_prop.fillna(0)


if inputs == 'ACS_MSA':
    df_msa1 = df_acs.groupby(['MSA', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()


    ## Dcasts
    
    # MSA
    df_msa2 = df_msa1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()

    
    
    # missing values represent a population of 0
    df_msa2 = df_msa2.fillna(0)

    
    if proportions == 'Yes':
        df_msa1['Proportion'] = df_msa1['Total'] / df_msa1[df_msa1['Variable'] != 'Total'].groupby(['MSA', 'Year'])['Total'].transform('sum')
        df_msa1['Proportion'] = df_msa1['Proportion'].fillna(1)

        df_msa2_prop = df_msa1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Proportion').reset_index()
        
        
        df_msa2_prop = df_msa2_prop.fillna(1)

df_acs.head()

In [ ]:
# Set output name
name_output_long = [indicator_name, ' ', geography, ' ', estimate, ' Long.xlsx']
name_output_wide = [indicator_name, ' ', geography, ' ', estimate, ' Wide.xlsx']

name_output_long = "".join(name_output_long)
name_output_wide = "".join(name_output_wide)

In [ ]:
if inputs == 'ACS_Tract':
    if proportions == 'Yes':
        # Export long
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
            df_acs      .to_excel(writer, index = False, sheet_name = 'Full Tracts')
            df_acs1     .to_excel(writer, index = False, sheet_name = 'Tracts'  )
            df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
            df_mpo1     .to_excel(writer, index = False, sheet_name = 'MPO'     )
        
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
            df_acs2          .to_excel(writer, index = False, sheet_name = 'Tracts Total'   )
            df_counties2     .to_excel(writer, index = False, sheet_name = 'Counties Total' )
            df_mpo2          .to_excel(writer, index = False, sheet_name = 'MPO Total'      )
            df_acs2_prop     .to_excel(writer, index = False, sheet_name = 'Tracts Prop'  )
            df_counties2_prop.to_excel(writer, index = False, sheet_name = 'Counties Prop')
            df_mpo2_prop     .to_excel(writer, index = False, sheet_name = 'MPO Prop'     )
    
    else:
        # Export long
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
            df_acs      .to_excel(writer, index = False, sheet_name = 'Full Tracts')
            df_acs1     .to_excel(writer, index = False, sheet_name = 'Tracts'  )
            df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
            df_mpo1     .to_excel(writer, index = False, sheet_name = 'MPO'     )
        
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
            df_acs2          .to_excel(writer, index = False, sheet_name = 'Tracts Total'   )
            df_counties2     .to_excel(writer, index = False, sheet_name = 'Counties Total' )
            df_mpo2          .to_excel(writer, index = False, sheet_name = 'MPO Total'      )


if inputs == 'ACS_MSA':
    if proportions == 'Yes':
    
        # Export long
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
            df_acs .to_excel(writer, index = False, sheet_name = 'Full MSA')
            df_msa1.to_excel(writer, index = False, sheet_name = 'MSA'     )
        
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
            df_msa2      .to_excel(writer, index = False, sheet_name = 'MSA Total')
            df_msa2_prop.to_excel(writer, index = False, sheet_name = 'MSA Total')
    else:
        # Export long
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
            df_acs .to_excel(writer, index = False, sheet_name = 'Full MSA')
            df_msa1.to_excel(writer, index = False, sheet_name = 'MSA'     )
        
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
            df_msa2.to_excel(writer, index = False, sheet_name = 'MSA Total')

if inputs == 'PUMS_PUMA':
    if proportions == 'Yes':
    
        # Export long
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
            df_acs .to_excel(writer, index = False, sheet_name = 'PUMS')
        
    else:
        # Export long
        with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
            df_acs .to_excel(writer, index = False, sheet_name = 'PUMS')
